In [ ]:
!pip install tabpfn tabpfn-extensions openml scikit-learn
import os; os.environ["TABPFN_TOKEN"] = ""

In [ ]:
# =====================================================================
# 1. INITIAL SYSTEM IMPORTS & CONFIGURATION
# =====================================================================
import os
import io
import time
import json
import gc
import traceback
from datetime import datetime
import random
import numpy as np
import openml
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, 
    roc_auc_score, 
    log_loss, 
    mean_squared_error, 
    root_mean_squared_error, 
    r2_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn_extensions.embedding import TabPFNEmbedding

# --- GOOGLE DRIVE INTEGRATION IMPORTS ---
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

global_start_time = time.time()
SUMMARY_FILENAME = "summary.csv"
GDRIVE_FOLDER_ID = "1Dmopi2d5fl16fDKrBBRmmdslPU10_bGj"

# --- OAUTH CONFIGURATION ---
SCOPES = ['https://www.googleapis.com/auth/drive.file']

# Live Credentials directly embedded as a Python dictionary
TOKEN_INFO = {
    "token": "",
    "refresh_token": "",
    "token_uri": "https://oauth2.googleapis.com/token",
    "client_id": "",
    "client_secret": "",
    "scopes": ["https://www.googleapis.com/auth/drive.file"],
    "universe_domain": "googleapis.com",
    "account": ""
}

def get_creds():
    # Load credentials strictly from memory using the dictionary above
    creds = Credentials.from_authorized_user_info(TOKEN_INFO, SCOPES)
    
    # If the token is expired, this seamlessly pings Google for a fresh token
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request()) # Refreshes silently without printing
        
    return creds

def sync_summary_from_drive(filename, folder_id):
    """Downloads the existing summary file from Google Drive to establish a Source of Truth."""
    try:
        creds = get_creds()
        service = build('drive', 'v3', credentials=creds)
        query = f"name='{filename}' and '{folder_id}' in parents and trashed=false"
        results = service.files().list(q=query, spaces='drive', fields='files(id, name)').execute()
        items = results.get('files', [])
        
        if items:
            file_id = items[0]['id']
            request = service.files().get_media(fileId=file_id)
            fh = io.FileIO(filename, 'wb')
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while done is False:
                status, done = downloader.next_chunk()
            print(f"[DRIVE SYNC] Downloaded existing '{filename}' from Drive as Source of Truth.")
        else:
            print(f"[DRIVE SYNC] No existing '{filename}' found in Drive. Starting fresh.")
    except Exception as e:
        print(f"[DRIVE SYNC ERROR] Could not sync {filename}: {str(e)}")

def upload_to_drive(file_path, folder_id, max_retries=3):
    """Uploads files to Google Drive with exponential backoff for network resilience."""
    filename = os.path.basename(file_path)

    for attempt in range(max_retries):
        try:
            creds = get_creds()
            service = build('drive', 'v3', credentials=creds)

            # 1. Search for existing file
            query = f"name='{filename}' and '{folder_id}' in parents and trashed=false"
            results = service.files().list(q=query, spaces='drive', fields='files(id, name)').execute()
            items = results.get('files', [])

            media = MediaFileUpload(file_path, resumable=True)

            # 2. Update or Create
            if items:
                file_id = items[0]['id']
                file = service.files().update(fileId=file_id, media_body=media).execute()
                print(f"[DRIVE SUCCESS] Updated existing file: {file.get('id')}")
            else:
                file_metadata = {'name': filename, 'parents': [folder_id]}
                file = service.files().create(body=file_metadata, media_body=media, fields='id').execute()
                print(f"[DRIVE SUCCESS] Created new file: {file.get('id')}")

            return # Exit the function if successful

        except Exception as e:
            print(f"[DRIVE WARNING] Attempt {attempt + 1}/{max_retries} failed for {filename}. Error: {str(e)}")
            if attempt < max_retries - 1:
                time.sleep(5 * (attempt + 1)) # Sleeps for 5s, then 10s, etc.
            else:
                print(f"[DRIVE CRITICAL ERROR] Completely failed to upload {filename} after {max_retries} attempts.")

# --- BOOT SEQUENCE: ESTABLISH SOURCE OF TRUTH ---
print("\n" + "="*70)
print("BOOTING PIPELINE & SYNCING WITH GOOGLE DRIVE")
print("="*70)
sync_summary_from_drive(SUMMARY_FILENAME, GDRIVE_FOLDER_ID)

# TARGET_DATASETS = [
#     {"id": 46980, "task": "multiclass"},   # MIC
#     {"id": 46915, "task": "binary"},       # churn
#     {"id": 46964, "task": "regression"},   # wine_quality
# ]

# --- KAGGLE GLOBAL CONFIGURATION ---
TARGET_DATASETS = [
    {"id": 46913, "task": "binary"},       # blood-transfusion-service-center
    {"id": 46921, "task": "binary"},       # diabetes
    {"id": 46906, "task": "multiclass"},   # anneal
    {"id": 46954, "task": "regression"},   # QSAR_fish_toxicity
    {"id": 46918, "task": "binary"},       # credit-g
    {"id": 46941, "task": "multiclass"},   # maternal_health_risk
    {"id": 46917, "task": "regression"},   # concrete_compressive_strength
    {"id": 46952, "task": "binary"},       # qsar-biodeg
    {"id": 46931, "task": "regression"},   # healthcare_insurance_expenses
    {"id": 46963, "task": "multiclass"},   # website_phishing
    {"id": 46927, "task": "binary"},       # Fitness_Club
    {"id": 46904, "task": "regression"},   # airfoil_self_noise
    {"id": 46907, "task": "regression"},   # Another-Dataset-on-used-Fiat-500
    {"id": 46980, "task": "multiclass"},   # MIC
    {"id": 46938, "task": "binary"},       # Is-this-a-good-customer
    {"id": 46940, "task": "binary"},       # Marketing_Campaign
    {"id": 46930, "task": "binary"},       # hazelnut-spread-contaminant-detection
    {"id": 46956, "task": "binary"},       # seismic-bumps
    {"id": 46958, "task": "multiclass"},   # splice
    {"id": 46912, "task": "binary"},       # Bioresponse
    {"id": 46933, "task": "multiclass"},   # hiva_agnostic
    {"id": 46960, "task": "multiclass"},   # students_dropout_and_academic_success
    {"id": 46915, "task": "binary"},       # churn
    {"id": 46953, "task": "regression"},   # QSAR-TID-11
    {"id": 46950, "task": "binary"},       # polish_companies_bankruptcy
    {"id": 46964, "task": "regression"},   # wine_quality
    {"id": 46962, "task": "binary"},       # taiwanese_bankruptcy_prediction
    {"id": 46969, "task": "binary"},       # NATICUSdroid
    {"id": 46916, "task": "binary"},       # coil2000_insurance_policies
    {"id": 46911, "task": "binary"},       # Bank_Customer_Churn
    {"id": 46932, "task": "binary"},       # heloc
    {"id": 46979, "task": "binary"},       # jm1
    {"id": 46924, "task": "binary"},       # E-CommereShippingData
    {"id": 46947, "task": "binary"},       # online_shoppers_intention
    {"id": 46937, "task": "binary"},       # in_vehicle_coupon_recommendation
    {"id": 46942, "task": "regression"},   # miami_housing
    {"id": 46935, "task": "binary"},       # HR_Analytics_Job_Change_of_Data_Scientists
    {"id": 46934, "task": "regression"},   # houses
    {"id": 46961, "task": "regression"},   # superconductivity
    {"id": 46919, "task": "binary"},       # credit_card_clients_default
    {"id": 46905, "task": "binary"},       # Amazon_employee_access
    {"id": 46910, "task": "binary"},       # bank-marketing
    {"id": 46928, "task": "regression"},   # Food_Delivery_Time
    {"id": 46949, "task": "regression"},   # physiochemical_protein
    {"id": 46939, "task": "binary"},       # kddcup09_appetency
    {"id": 46923, "task": "regression"},   # diamonds
    {"id": 46922, "task": "binary"},       # Diabetes130US
    {"id": 46908, "task": "binary"},       # APSFailure
    {"id": 46955, "task": "multiclass"},   # SDSS17
    {"id": 46920, "task": "binary"},       # customer_satisfaction_in_airline
    {"id": 46929, "task": "binary"}        # GiveMeSomeCredit
]

INNER_FOLDS = 8 # Protects your downstream CPU tuning script's 8-fold CV
TEST_SIZE = 0.20 # Fixed holdout size for the single outer pass

timestamp_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# =====================================================================
# 2. MASTER ITERATION ENGINE (SINGLE PASS WITH FAULT TOLERANCE)
# =====================================================================
for idx, config in enumerate(TARGET_DATASETS):
    DATASET_ID = config["id"]
    TASK = config["task"]

    # --- DATASET LEVEL SKIP LOGIC ---
    dataset_completed = False
    if os.path.exists(SUMMARY_FILENAME):
        df_check = pd.read_csv(SUMMARY_FILENAME)
        if 'dataset_id' in df_check.columns:
            mask = (df_check['dataset_id'].astype(str) == str(DATASET_ID))
            if mask.any():
                dataset_completed = True

    if dataset_completed:
        print(f"\n[SKIP] Dataset ID {DATASET_ID} already logged. Skipping.")
        continue

    print("\n" + "#"*70)
    print(f"### [{idx + 1}/{len(TARGET_DATASETS)}] INITIATING PIPELINE FOR ID: {DATASET_ID} | TASK: {TASK.upper()}")
    print("#"*70)
    
    dataset_start_time = time.time()
    
    try:
        # --- SHARED DATA INGESTION ---
        print(f"Downloading target dataset from OpenML...")
        download_start = time.time()
        dataset = openml.datasets.get_dataset(DATASET_ID)
        X, y, _, _ = dataset.get_data(
            dataset_format="dataframe", target=dataset.default_target_attribute
        )
        
        # Defensive casting to prevent SparseDtype crashes in downstream PyTorch operations
        X = pd.DataFrame(X)
        
        dataset_name = dataset.name.lower().replace(" ", "_")
        download_time = time.time() - download_start

        print(f"Download completed in {download_time:.2f} seconds.")
        print(f"Dataset mapped to local name: '{dataset_name}'")

        # =====================================================================
        # --- DATASET CHARACTERIZATION ---
        # Calculates metadata for the academic paper (Table 1 profiling)
        # =====================================================================
        n_samples_total = X.shape[0]
        n_features_total = X.shape[1]
        missing_value_ratio = float(X.isna().sum().sum() / (n_samples_total * n_features_total))
        
        # Track target imbalance (Classification) or variance (Regression)
        target_metric_val = 0.0
        if TASK in ["binary", "multiclass"]:
            # Find the ratio of the absolute minority class
            class_counts = y.value_counts(normalize=True)
            target_metric_val = float(class_counts.min()) 
        else:
            # Calculate target variance for regression context
            target_metric_val = float(y.var())

        eval_start_time = time.time()
        print(f"\n{'='*60}")
        print(f"--- PROCESSING SINGLE PASS EVALUATION ({dataset_name}) ---")
        print(f"{'='*60}")

        # =====================================================================
        # 3. LEAKAGE PROTECTION & STRATIFIED SPLITTING
        # =====================================================================
        print(f"Executing deterministic {100-int(TEST_SIZE*100)}/{int(TEST_SIZE*100)} partition split...")
        
        if TASK in ["binary", "multiclass"]:
            X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
                X, y, test_size=TEST_SIZE, shuffle=True, stratify=y, random_state=42
            )
        else:
            X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
                X, y, test_size=TEST_SIZE, shuffle=True, random_state=42
            )
            
        # Reset indices to match old iloc behavior
        X_train_raw = X_train_raw.copy().reset_index(drop=True)
        X_test_raw = X_test_raw.copy().reset_index(drop=True)
        y_train_raw = y_train_raw.copy().reset_index(drop=True)
        y_test_raw = y_test_raw.copy().reset_index(drop=True)

        # --- POST-SPLIT THRESHOLD FILTERING FOR CLASSIFICATION DATA ---
        if TASK in ["binary", "multiclass"]:
            # Safely evaluate class counts directly on the training slice
            train_counts = y_train_raw.value_counts()
            rare_train_classes = train_counts[train_counts < INNER_FOLDS].index.tolist()
            
            if rare_train_classes:
                print(f"\n[DEFENSIVE WARNING] Training categories {rare_train_classes} contain fewer than {INNER_FOLDS} samples due to split ratios.")
                print("Pruning rare rows from train and test slices to guarantee clean stratification...")
                
                # Prune from training set
                train_keep_mask = ~y_train_raw.isin(rare_train_classes)
                X_train_raw = X_train_raw[train_keep_mask].reset_index(drop=True)
                y_train_raw = y_train_raw[train_keep_mask].reset_index(drop=True)
                
                # Prune from test set to keep classes perfectly synchronized
                test_keep_mask = ~y_test_raw.isin(rare_train_classes)
                X_test_raw = X_test_raw[test_keep_mask].reset_index(drop=True)
                y_test_raw = y_test_raw[test_keep_mask].reset_index(drop=True)
                
                print(f"Partitions sanitized. New Train Shape: {X_train_raw.shape}, New Test Shape: {X_test_raw.shape}")

        # Parse high-level schema metadata vectors for downstream dataframe assembly
        feature_names = np.array(list(X_train_raw.columns), dtype=str)
        cat_features = np.array(
            list(X_train_raw.select_dtypes(["object", "category", "bool"]).columns),
            dtype=str,
        )

        # =====================================================================
        # 4. BASELINE EVALUATION (Standalone TabPFN Performance)
        # =====================================================================
        print(f"\n--- Running Baseline TabPFN {TASK.capitalize()} Evaluation ---")

        # Process targets based on inferred task
        if TASK in ["binary", "multiclass"]:
            le = LabelEncoder()
            # FIT ON BOTH to prevent unseen label errors in the test set if the split isolates a class
            # Hard coerce to string to avoid TypeError with mixed dtypes/categorical objects
            y_combined = pd.concat([y_train_raw, y_test_raw]).astype(str)
            le.fit(y_combined)
            y_train_proc = le.transform(y_train_raw.astype(str))
            y_test_proc = le.transform(y_test_raw.astype(str))
            
            print("Training TabPFNClassifier on GPU...")
            model = TabPFNClassifier(device="cuda", n_estimators=8)
        else:
            y_train_proc = y_train_raw.to_numpy(dtype=np.float64)
            y_test_proc = y_test_raw.to_numpy(dtype=np.float64)
            
            print("Training TabPFNRegressor on GPU...")
            model = TabPFNRegressor(device="cuda", n_estimators=8)

        train_start_time = time.time()
        model.fit(X_train_raw, y_train_proc)
        train_duration = time.time() - train_start_time

        print("Generating target inference predictions...")
        infer_start_time = time.time()
        
        if TASK == "binary":
            y_pred = model.predict(X_test_raw)
            y_proba = model.predict_proba(X_test_raw)[:, 1]
            
            # TabArena Standard: Evaluate errors (lower is better)
            primary_metric_name = "1-AUROC"
            primary_metric_val = 1.0 - roc_auc_score(y_test_proc, y_proba)
            
            secondary_metric_name, secondary_metric_val = "Accuracy", accuracy_score(y_test_proc, y_pred)

        elif TASK == "multiclass":
            y_pred = model.predict(X_test_raw)
            y_proba = model.predict_proba(X_test_raw)
            all_classes = np.unique(y_train_proc)
            
            # Log Loss is inherently an error metric (lower is better)
            primary_metric_name, primary_metric_val = "Log_Loss", log_loss(y_test_proc, y_proba, labels=all_classes)
            secondary_metric_name, secondary_metric_val = "Accuracy", accuracy_score(y_test_proc, y_pred)

        else:
            y_pred = model.predict(X_test_raw)
            
            # RMSE is inherently an error metric (lower is better)
            primary_metric_name, primary_metric_val = "RMSE", root_mean_squared_error(y_test_proc, y_pred)
            secondary_metric_name, secondary_metric_val = "R2_Score", r2_score(y_test_proc, y_pred)

        infer_duration = time.time() - infer_start_time
        
        print(f"TabPFN Baseline {primary_metric_name}: {primary_metric_val:.4f}")
        print(f"TabPFN Baseline {secondary_metric_name}: {secondary_metric_val:.4f}")
        print(f"Baseline training completed in {train_duration:.2f} seconds.")
        print(f"Baseline inference completed in {infer_duration:.2f} seconds.")

        # =====================================================================
        # 5. REPRESENTATION EXTRACTION (TabPFN Deep Text Embeddings)
        # =====================================================================
        print("\n--- Running Latent Space Embedding Extraction ---")
        embed_start_time = time.time()
        print("Initializing TabPFN embedder...")

        # Construct a clean, task-matched base model configuration for the embedder
        if TASK in ["binary", "multiclass"]:
            embedding_base = TabPFNClassifier(device="cuda", n_estimators=8)
        else:
            embedding_base = TabPFNRegressor(device="cuda", n_estimators=8)

        # Pass the base model explicitly to freeze the task and match ensemble depth
        embedder = TabPFNEmbedding(model=embedding_base, n_fold=INNER_FOLDS, shuffle=True, random_state=42)

        print("Extracting training matrix embeddings (this may take a moment)...")
        train_embeds_3d = embedder.fit_transform(X_train_raw, y_train_proc)

        print("Extracting testing matrix embeddings...")
        test_embeds_3d = embedder.transform(X_test_raw)

        # Convert to pure numpy arrays before checking shape logic
        train_embeds_3d = np.array(train_embeds_3d)
        test_embeds_3d = np.array(test_embeds_3d)

        # Safely handle potential 2D or 3D outputs from the TabPFN ensemble wrapper
        if len(train_embeds_3d.shape) == 3:
            # SHAPE: (n_estimators, n_samples, embedding_dim)
            X_train_embeds = train_embeds_3d.mean(axis=0).astype(np.float32)
            X_test_embeds = test_embeds_3d.mean(axis=0).astype(np.float32)
        else:
            # Array is already collapsed to 2D (n_samples, embedding_dim)
            X_train_embeds = train_embeds_3d.astype(np.float32)
            X_test_embeds = test_embeds_3d.astype(np.float32)

        embed_duration = time.time() - embed_start_time
        print(f"Embedding extraction completed in {embed_duration:.2f} seconds.")

        # =====================================================================
        # 6. COMPREHENSIVE ARCHIVE COMPRESSION EXPORT
        # =====================================================================
        save_start_time = time.time()
        
        # Calculate TOTAL time since OpenML fetch started
        dataset_total_time = time.time() - dataset_start_time
        
        output_filename = f"{dataset_name}.npz"
        print(f"\nSaving comprehensive self-contained archive to disk as '{output_filename}'...")

        np.savez_compressed(
            output_filename,
            # Main Model Modeling Matrix Inputs
            X_train_embed=X_train_embeds,
            X_test_embed=X_test_embeds,
            X_train_raw=X_train_raw.to_numpy(dtype=object),
            X_test_raw=X_test_raw.to_numpy(dtype=object),
            y_train=y_train_raw.to_numpy(dtype=object),
            y_test=y_test_raw.to_numpy(dtype=object),
            
            # Dataset Schema Metadata Profiles
            feature_names=feature_names,
            cat_features=cat_features,
            missing_value_ratio=np.array(missing_value_ratio, dtype=np.float64),
            target_profile_val=np.array(target_metric_val, dtype=np.float64), 
            
            # Baseline Model Tracking Performance Attributes (Dynamic)
            baseline_primary_metric_name=np.array(primary_metric_name),
            baseline_primary_metric_val=np.array(primary_metric_val, dtype=np.float64),
            baseline_secondary_metric_name=np.array(secondary_metric_name),
            baseline_secondary_metric_val=np.array(secondary_metric_val, dtype=np.float64),
            
            # Operational Documentation & Scalar Metrics
            task_type=np.array(TASK),
            dataset_name=np.array(dataset_name),
            dataset_id=np.array(DATASET_ID, dtype=np.int64),
            n_samples=np.array(X_train_raw.shape[0] + X_test_raw.shape[0], dtype=np.int64),
            n_features_raw=np.array(X_train_raw.shape[1], dtype=np.int64),
            n_embedding_dim=np.array(X_train_embeds.shape[1], dtype=np.int64),
            timestamp=np.array(timestamp_str),

            # Telemetry / Execution Times
            time_train_sec=np.array(train_duration, dtype=np.float64),
            time_inference_sec=np.array(infer_duration, dtype=np.float64),
            time_embedding_sec=np.array(embed_duration, dtype=np.float64),
            time_total_sec=np.array(dataset_total_time, dtype=np.float64),
        )
        
        save_duration = time.time() - save_start_time
        print(f"Disk archiving completed in {save_duration:.2f} seconds.")
        print(f"Total processing time for {dataset_name}: {dataset_total_time:.2f} seconds.")

        # Push to Google Drive immediately after saving
        upload_to_drive(output_filename, GDRIVE_FOLDER_ID)

        # Log telemetry for the live receipt
        summary_record = {
            "dataset": dataset_name,
            "dataset_id": DATASET_ID,
            "task_type": TASK,
            
            # Performance Data
            "primary_metric": primary_metric_name,
            "baseline_primary_score": primary_metric_val,
            "secondary_metric": secondary_metric_name,
            "baseline_secondary_score": secondary_metric_val,
            
            # Timing Data
            "time_train_sec": train_duration,
            "time_inference_sec": infer_duration,
            "time_embedding_sec": embed_duration,
            "time_total_sec": dataset_total_time,
        }

        # =====================================================================
        # 7. LIVE CHECKPOINT: OVERWRITE RECEIPT AFTER EVERY DATASET
        # =====================================================================
        current_dataset_record = pd.DataFrame([summary_record])
        
        if os.path.exists(SUMMARY_FILENAME):
            global_df = pd.read_csv(SUMMARY_FILENAME)
            # Remove this dataset if it exists to avoid duplication on re-runs
            if 'dataset_id' in global_df.columns:
                mask = (global_df['dataset_id'].astype(str) == str(DATASET_ID))
                global_df = global_df[~mask]
            global_df = pd.concat([global_df, current_dataset_record], ignore_index=True)
        else:
            global_df = current_dataset_record
            
        global_df.to_csv(SUMMARY_FILENAME, index=False)
        upload_to_drive(SUMMARY_FILENAME, GDRIVE_FOLDER_ID)
        print(f"Live GPU Execution receipt updated and uploaded: '{SUMMARY_FILENAME}'.")

    # --- CRITICAL FAILURE CATCHER ---
    except Exception as e:
        print("\n" + "!"*70)
        print(f"[PIPELINE FATAL ERROR] Dataset ID {DATASET_ID} crashed during execution.")
        print("Error Traceback:")
        traceback.print_exc()
        print("!"*70)
        
        # Ensure we log the failure so we know it was skipped due to an error, not just missing
        error_record = {
            "dataset": f"FAILED_ID_{DATASET_ID}",
            "dataset_id": DATASET_ID,
            "task_type": TASK,
            "primary_metric": "ERROR",
            "baseline_primary_score": 0.0,
            "secondary_metric": "ERROR",
            "baseline_secondary_score": 0.0,
            "time_train_sec": 0.0,
            "time_inference_sec": 0.0,
            "time_embedding_sec": 0.0,
            "time_total_sec": 0.0
        }
        
        err_df = pd.DataFrame([error_record])
        if os.path.exists(SUMMARY_FILENAME):
            global_df = pd.read_csv(SUMMARY_FILENAME)
            global_df = pd.concat([global_df, err_df], ignore_index=True)
        else:
            global_df = err_df
            
        global_df.to_csv(SUMMARY_FILENAME, index=False)
        upload_to_drive(SUMMARY_FILENAME, GDRIVE_FOLDER_ID)
        
        print(f"[RECOVERY PROTOCOL] Safely moving to next dataset in queue...")
        continue # Jump straight to the top of the loop for the next dataset

    # --- ISOLATED MEMORY FLUSH (Executes on Success OR Failure) ---
    finally:
        # Explicitly delete large memory hogs to prevent Kaggle OOM crashes
        # Variables might not exist if the loop aborted early, so we suppress NameErrors
        for var in ['X', 'y', 'X_train_raw', 'X_test_raw', 'y_train_raw', 'y_test_raw', 
                    'train_embeds_3d', 'test_embeds_3d', 'X_train_embeds', 'X_test_embeds', 
                    'model', 'embedder', 'dataset']:
            try:
                del locals()[var]
            except KeyError:
                pass 
        
        gc.collect()
        torch.cuda.empty_cache() # Flushes the GPU VRAM
        print("[SYSTEM] VRAM and CPU Memory flushed. Ready for next block.")

# =====================================================================
# 8. GLOBAL SCRIPT COMPLETION
# =====================================================================
global_duration = time.time() - global_start_time
print("\n" + "="*70)
print(f"ALL DATASETS PROCESSED!")
print(f"Total Batch GPU Execution Time: {global_duration:.2f} seconds ({global_duration/60:.2f} minutes).")
print(f"Final Global Receipt: '{SUMMARY_FILENAME}' synced to Google Drive.")
print("="*70)

In [ ]:
# # GENERATE A NEW GOOGLE DRIVE TOKEN
# # YOU NEED TO UPLOAD THE client_secret.json BY IN KAGGLE BY SELECTING SIDEBAR > INPUT> UPLOAD > NEW MODEL

# from google_auth_oauthlib.flow import InstalledAppFlow

# # Paste your client secret file path here
# SECRET_PATH = '/kaggle/input/datasets/gusalbukrk/gdrive/client_secret.json'
# SCOPES = ['https://www.googleapis.com/auth/drive.file']

# def generate_new_json():
#     flow = InstalledAppFlow.from_client_secrets_file(SECRET_PATH, SCOPES)
#     flow.redirect_uri = 'http://localhost'
    
#     auth_url, _ = flow.authorization_url(access_type='offline', include_granted_scopes='true')
    
#     print(f"1. Open this URL in your browser and authorize the app:\n\n{auth_url}\n")
#     code = input("2. Enter the authorization code from the browser's address bar: ")
    
#     flow.fetch_token(code=code)
#     creds = flow.credentials
    
#     print("\n" + "="*60)
#     print("SUCCESS! COPY THIS NEW JSON AND PASTE IT INTO YOUR 'TOKEN_INFO' DICTIONARY:")
#     print("="*60)
#     print(creds.to_json())
#     print("="*60)

# generate_new_json()